## Demo — EU AI Act Risk Tier Classifier

## Scenario: AutoPilot Logistics

AutoPilot Logistics is a European logistics company with four AI systems:

- **RouteGenius** — GenAI route optimization (plans delivery routes, predicts traffic)
- **WarehouseWatch** — AI-powered worker safety monitoring (camera-based unsafe-behavior detection)
- **HireBot** — AI applicant screening for warehouse positions
- **AssistGPT** — Foundation-model API integrated into the dispatcher console for natural-language ops queries (a General-Purpose AI / GPAI deployment under EU AI Act Articles 51–55)

In this demo, the instructor builds a rule-based classifier that determines the EU AI Act risk tier for each system based on its characteristics. The classifier is the durable artifact — every new AI system in the pipeline gets classified the same way.


In [1]:
import pandas as pd

print("EU AI Act Risk Tier Classifier — Demo")
print("Scenario: AutoPilot Logistics")
print("=" * 50)

EU AI Act Risk Tier Classifier — Demo
Scenario: AutoPilot Logistics


## Step 1: Define the Risk Tiers

The EU AI Act defines five risk tiers in the current framework. The GPAI tier (Articles 51–55) was introduced by Regulation (EU) 2024/1689 — without it, GPT-class systems are silently classified as Limited, which is incorrect under the current Act.

In [2]:
# 5-tier EU AI Act framework — GPAI per Regulation (EU) 2024/1689
RISK_TIERS = {
    "Unacceptable": {
        "description": "Prohibited AI practices (Article 5)",
        "examples": ["Social scoring", "Manipulation of vulnerable groups"],
        "articles": ["Article 5"],
    },
    "High-Risk": {
        "description": "Significant impact on fundamental rights or safety (Annex III)",
        "examples": ["Recruitment screening", "Biometric identification", "Worker monitoring"],
        "articles": [
            "Article 6", "Article 9", "Article 10", "Article 11",
            "Article 13", "Article 14", "Article 15", "Article 43",
            "Article 73 (serious-incident reporting — enforces Aug 2026)",
        ],
    },
    "GPAI": {
        "description": "General-Purpose AI / foundation-model deployments (Reg. (EU) 2024/1689)",
        "examples": ["Foundation-model APIs in customer-facing assistants", "RAG over GPT-class models"],
        "articles": [
            "Article 51 (classification)", "Article 52 (procedure)",
            "Article 53 (provider obligations)", "Article 54 (authorised reps)",
            "Article 55 (systemic-risk GPAI)",
        ],
    },
    "Limited": {
        "description": "Transparency obligations (Article 50)",
        "examples": ["Non-GPAI chatbots", "Content recommenders"],
        "articles": ["Article 50"],
    },
    "Minimal": {
        "description": "Standard / voluntary code of conduct",
        "examples": ["Spam filters", "Internal supply-chain ML"],
        "articles": ["General transparency"],
    },
}

for tier, info in RISK_TIERS.items():
    print(f"\n{tier}: {info['description']}")
    print(f"  Articles: {', '.join(info['articles'])}")


Unacceptable: Prohibited AI practices (Article 5)
  Articles: Article 5

High-Risk: Significant impact on fundamental rights or safety (Annex III)
  Articles: Article 6, Article 9, Article 10, Article 11, Article 13, Article 14, Article 15, Article 43, Article 73 (serious-incident reporting — enforces Aug 2026)

GPAI: General-Purpose AI / foundation-model deployments (Reg. (EU) 2024/1689)
  Articles: Article 51 (classification), Article 52 (procedure), Article 53 (provider obligations), Article 54 (authorised reps), Article 55 (systemic-risk GPAI)

Limited: Transparency obligations (Article 50)
  Articles: Article 50

Minimal: Standard / voluntary code of conduct
  Articles: General transparency


## Step 2: Build the Classifier

A rule-based function takes a system's characteristics and returns its tier, applicable articles, required actions, and the reasoning chain. The order of rules matters — Unacceptable fires first; High-Risk and GPAI come before Limited so that GPT-class systems route correctly to GPAI rather than falling through to Limited.

In [3]:
def classify_eu_ai_act(system_name, characteristics):
    """Classify an AI system under the EU AI Act.

    Returns dict with: system, tier, reasoning (list), required_actions (list),
    applicable_articles (list).
    """
    reasons = []
    articles = []

    # Rule 1: Unacceptable (Article 5)
    if characteristics.get("social_scoring", False):
        return {
            "system": system_name,
            "tier": "Unacceptable",
            "reasoning": ["Performs social scoring — prohibited under Article 5"],
            "required_actions": ["MUST NOT BE DEPLOYED. Remove immediately."],
            "applicable_articles": ["Article 5"],
        }

    # Rule 2: High-Risk (Article 6 + Annex III)
    if characteristics.get("affects_employment", False):
        reasons.append("Makes or influences employment decisions (Annex III, area 4)")
        articles.extend(["Article 6", "Annex III(4)"])
    if characteristics.get("monitors_workers", False):
        reasons.append("Monitors workers in the workplace (Annex III, area 4)")
        articles.extend(["Article 6", "Annex III(4)"])
    if characteristics.get("uses_biometric", False):
        reasons.append("Uses biometric data processing (Annex III, area 1)")
        articles.extend(["Article 6", "Annex III(1)"])
    if characteristics.get("is_safety_critical", False):
        reasons.append("Safety-critical system (Annex III, area 2)")
        articles.extend(["Article 6", "Annex III(2)"])
    if characteristics.get("impacts_fundamental_rights", False):
        reasons.append("Impacts fundamental rights of individuals")
        articles.append("Article 6")

    if reasons:
        return {
            "system": system_name,
            "tier": "High-Risk",
            "reasoning": reasons,
            "required_actions": [
                "Conduct conformity assessment (Article 43)",
                "Implement risk-management system (Article 9)",
                "Ensure data governance (Article 10)",
                "Maintain technical documentation (Article 11)",
                "Enable human oversight (Article 14)",
                "Ensure accuracy and robustness (Article 15)",
                "Stand up Article 73 serious-incident reporting workflow (enforces Aug 2026)",
                "Register in EU database (Article 49)",
            ],
            "applicable_articles": list(dict.fromkeys(articles + ["Article 73"])),
        }

    # Rule 3: GPAI — fires BEFORE Limited so GPT-class systems route correctly
    if characteristics.get("is_general_purpose_foundation_model", False):
        gpai_articles = [
            "Article 51", "Article 52", "Article 53", "Article 54",
        ]
        gpai_actions = [
            "Transparency to downstream deployers (Art. 53)",
            "Copyright disclosure of training-data sources",
            "Technical documentation per Annex XI",
        ]
        if characteristics.get("has_systemic_risk", False):
            gpai_articles.append("Article 55")
            gpai_actions.append("Model evaluation + adversarial testing + serious-incident reporting (Art. 55)")
        return {
            "system": system_name,
            "tier": "GPAI",
            "reasoning": ["General-purpose foundation-model deployment — Articles 51–55"],
            "required_actions": gpai_actions,
            "applicable_articles": gpai_articles,
        }

    # Rule 4: Limited (transparency)
    if characteristics.get("interacts_with_users", False):
        return {
            "system": system_name,
            "tier": "Limited",
            "reasoning": ["Interacts directly with users — transparency obligation applies"],
            "required_actions": ["Disclose AI involvement to users (Article 50)"],
            "applicable_articles": ["Article 50"],
        }

    # Rule 5: Minimal (default)
    return {
        "system": system_name,
        "tier": "Minimal",
        "reasoning": ["No high-risk triggers identified"],
        "required_actions": ["No mandatory requirements; consider voluntary code of conduct."],
        "applicable_articles": [],
    }

## Step 3: Define the Four AI Systems

Each of AutoPilot Logistics' four systems is described as a dict of characteristics the classifier expects. AssistGPT is the GPAI case — note the `is_general_purpose_foundation_model` flag that routes it correctly.

In [4]:
autopilot_systems = {
    "RouteGenius": {
        "description": "GenAI route optimization — plans delivery routes, predicts traffic",
        "uses_biometric": False,
        "affects_employment": False,
        "monitors_workers": False,
        "impacts_fundamental_rights": False,
        "is_safety_critical": False,
        "social_scoring": False,
        "interacts_with_users": False,
        "is_general_purpose_foundation_model": False,
    },
    "WarehouseWatch": {
        "description": "AI worker-safety monitoring — camera-based unsafe-behavior detection",
        "uses_biometric": True,
        "affects_employment": False,
        "monitors_workers": True,
        "impacts_fundamental_rights": True,
        "is_safety_critical": True,
        "social_scoring": False,
        "interacts_with_users": False,
        "is_general_purpose_foundation_model": False,
    },
    "HireBot": {
        "description": "AI applicant screening — ranks job applicants for warehouse positions",
        "uses_biometric": False,
        "affects_employment": True,
        "monitors_workers": False,
        "impacts_fundamental_rights": True,
        "is_safety_critical": False,
        "social_scoring": False,
        "interacts_with_users": True,
        "is_general_purpose_foundation_model": False,
    },
    "AssistGPT": {
        "description": "Foundation-model API integrated into dispatcher console for natural-language ops queries",
        "uses_biometric": False,
        "affects_employment": False,
        "monitors_workers": False,
        "impacts_fundamental_rights": False,
        "is_safety_critical": False,
        "social_scoring": False,
        "interacts_with_users": True,
        "is_general_purpose_foundation_model": True,
        "has_systemic_risk": False,  # Below the Art. 51 systemic-risk compute threshold
    },
}

print(f"Defined {len(autopilot_systems)} AI systems for classification:")
for name, chars in autopilot_systems.items():
    print(f"  - {name}: {chars['description']}")

Defined 4 AI systems for classification:
  - RouteGenius: GenAI route optimization — plans delivery routes, predicts traffic
  - WarehouseWatch: AI worker-safety monitoring — camera-based unsafe-behavior detection
  - HireBot: AI applicant screening — ranks job applicants for warehouse positions
  - AssistGPT: Foundation-model API integrated into dispatcher console for natural-language ops queries


## Step 4: Run the Classifier

In [5]:
results = {name: classify_eu_ai_act(name, chars) for name, chars in autopilot_systems.items()}

for name, result in results.items():
    print(f"\n{'=' * 60}")
    print(f"System: {name}")
    print(f"Risk Tier: {result['tier']}")
    print(f"\nReasoning:")
    for r in result["reasoning"]:
        print(f"  - {r}")
    print(f"\nRequired Actions:")
    for a in result["required_actions"]:
        print(f"  - {a}")
    if result["applicable_articles"]:
        print(f"\nApplicable Articles: {', '.join(result['applicable_articles'])}")


System: RouteGenius
Risk Tier: Minimal

Reasoning:
  - No high-risk triggers identified

Required Actions:
  - No mandatory requirements; consider voluntary code of conduct.

System: WarehouseWatch
Risk Tier: High-Risk

Reasoning:
  - Monitors workers in the workplace (Annex III, area 4)
  - Uses biometric data processing (Annex III, area 1)
  - Safety-critical system (Annex III, area 2)
  - Impacts fundamental rights of individuals

Required Actions:
  - Conduct conformity assessment (Article 43)
  - Implement risk-management system (Article 9)
  - Ensure data governance (Article 10)
  - Maintain technical documentation (Article 11)
  - Enable human oversight (Article 14)
  - Ensure accuracy and robustness (Article 15)
  - Stand up Article 73 serious-incident reporting workflow (enforces Aug 2026)
  - Register in EU database (Article 49)

Applicable Articles: Article 6, Annex III(4), Annex III(1), Annex III(2), Article 73

System: HireBot
Risk Tier: High-Risk

Reasoning:
  - Makes or

## Step 5: Summary Table

In [6]:
summary = []
for name, result in results.items():
    summary.append({
        "System": name,
        "Risk Tier": result["tier"],
        "# Reasons": len(result["reasoning"]),
        "# Required Actions": len(result["required_actions"]),
        "Key Articles": ", ".join(result["applicable_articles"]) if result["applicable_articles"] else "N/A",
    })

df_summary = pd.DataFrame(summary)
print("\nEU AI Act Classification Summary — AutoPilot Logistics")
print("=" * 80)
print(df_summary.to_string(index=False))

print("\n\nKey Takeaway:")
print("  RouteGenius   — Minimal       (no triggers; voluntary code of conduct only)")
print("  WarehouseWatch — High-Risk     (worker monitoring + biometric + safety-critical)")
print("  HireBot        — High-Risk     (employment screening + fundamental rights)")
print("  AssistGPT      — GPAI          (foundation-model deployment under Articles 51–55)")
print("\n  Same company, four systems, four different EU AI Act tiers — including the GPAI layer.")


EU AI Act Classification Summary — AutoPilot Logistics
        System Risk Tier  # Reasons  # Required Actions                                                    Key Articles
   RouteGenius   Minimal          1                   1                                                             N/A
WarehouseWatch High-Risk          4                   8 Article 6, Annex III(4), Annex III(1), Annex III(2), Article 73
       HireBot High-Risk          2                   8                             Article 6, Annex III(4), Article 73
     AssistGPT      GPAI          1                   3                  Article 51, Article 52, Article 53, Article 54


Key Takeaway:
  RouteGenius   — Minimal       (no triggers; voluntary code of conduct only)
  WarehouseWatch — High-Risk     (worker monitoring + biometric + safety-critical)
  HireBot        — High-Risk     (employment screening + fundamental rights)
  AssistGPT      — GPAI          (foundation-model deployment under Articles 51–55)

  Sam

## Key Takeaways

- The EU AI Act uses a **risk-based approach** — not all AI systems face the same requirements.
- **Context matters**: RouteGenius optimizes routes (Minimal), but WarehouseWatch uses similar tech (camera + AI) for worker monitoring and becomes High-Risk.
- **High-Risk triggers** include: employment decisions, biometric processing, worker monitoring, safety-critical systems, fundamental-rights impact.
- **GPAI is a distinct tier** — without an explicit GPAI rule that fires before Limited, foundation-model integrations get silently misclassified.
- **Article 73** serious-incident reporting enforces from August 2026; surface it in the matrix on day one of any High-Risk deployment.
- A small rule-based classifier captures the rules in code so the next system in the pipeline arrives pre-classified — Excel documents the decision, Python reproduces it.

## Reference Notes

A few specification details for cross-referencing the EU AI Act citations used in this lesson:

- **GPAI tier provenance.** Articles 51–55 (GPAI) were introduced in Regulation (EU) 2024/1689 itself when adopted in June 2024. The substantive GPAI obligations became applicable on 2 August 2025. The broader high-risk regime becomes generally applicable on 2 August 2026; that date concerns enforcement of the wider regime, not the introduction of GPAI.
- **Article 10 vs Article 15.** The bias-prevention obligation sits in **Article 10(2)(f)–(g)** (data and data governance), not Article 15 (which covers accuracy, robustness, cybersecurity). The Compliance Matrix solution has been updated to reflect this.
- **Article 13 vs Article 50.** Article 13 governs provider-to-deployer transparency for high-risk systems. End-user / affected-person disclosure (chatbots, deepfakes, biometric/emotion-recognition notice) sits in **Article 50**; both can apply in tandem when a high-risk system is also user-facing.
- **Article 73 deadlines are tiered.** 15 days is the general default; **2 days** for widespread-infringement incidents (Art. 3(49)(b)); **10 days** for incidents involving death.
- **Annex I vs Annex III.** Annex III lists eight specific use-case areas; Annex I captures AI that is also a regulated product (or safety component) under product-safety legislation (EU MDR / IVDR, Machinery Regulation, etc.). Camera-based worker-safety systems and clinical decision support typically route through Annex I; pure-software AI for credit / hiring / biometrics routes through Annex III.